In [1]:
!pip install -q openneuro-py mne pyyaml

In [2]:
import os, pathlib
import pandas as pd
import openneuro as on

DATA_DIR = "/kaggle/working/data/ds004752"
os.makedirs(DATA_DIR, exist_ok=True)
on.download(dataset="ds004752", target_dir=DATA_DIR,
            include=["README*", "CHANGES", "dataset_description.json",
                     "participants.*", "*.tsv", "*.json"])

🌍 Preparing to download ds004752 …
📥 Checking 768 files, downloading as needed (5 concurrent downloads). 


Output()

✅ Finished downloading ds004752 (downloaded 768 files and 1.4 MB).


In [3]:
root = pathlib.Path(DATA_DIR)
failed = []
for sub in sorted(p.name for p in root.glob("sub-*")):
    folders = [f"{sub}/{s.name}/ieeg" for s in sorted((root / sub).glob("ses-*"))]
    try:
        on.download(dataset="ds004752", target_dir=DATA_DIR,
                    include=folders, max_retries=5, max_concurrent_downloads=4)
        print("OK", sub, len(folders))
    except Exception as e:
        print("FAILED", sub, repr(e)[:200]); failed.append(sub)
print("Failed:", failed)

🌍 Preparing to download ds004752 …
📥 Checking 33 files, downloading as needed (4 concurrent downloads). 


Output()

✅ Finished downloading ds004752 (downloaded 4 files and 293.0 MB).
OK sub-01 4
🌍 Preparing to download ds004752 …
📥 Checking 54 files, downloading as needed (4 concurrent downloads). 


Output()

✅ Finished downloading ds004752 (downloaded 7 files and 662.2 MB).
OK sub-02 7
🌍 Preparing to download ds004752 …
📥 Checking 26 files, downloading as needed (4 concurrent downloads). 


Output()

... (21 outputs left out)

Output()

✅ Finished downloading ds004752 (downloaded 8 files and 1.5 GB).
OK sub-14 8
🌍 Preparing to download ds004752 …
📥 Checking 33 files, downloading as needed (4 concurrent downloads). 


Output()

✅ Finished downloading ds004752 (downloaded 4 files and 800.1 MB).
OK sub-15 4
Failed: []


In [4]:
edfs = sorted(root.glob("sub-*/ses-*/ieeg/*_ieeg.edf"))
sizes = [p.stat().st_size for p in edfs]
print("EDF files:", len(edfs), "(expect 68)")
print("Total GB:", round(sum(sizes)/1e9, 2), "(expect ~8.8)")
print("Tiny files:", sum(s < 10_000_000 for s in sizes))

EDF files: 68 (expect 68)
Total GB: 8.84 (expect ~8.8)
Tiny files: 0


In [5]:
os.makedirs("/kaggle/working/meta", exist_ok=True)
def stack(pattern):
    rows = []
    for f in sorted(root.glob(pattern)):
        d = pd.read_csv(f, sep="\t")
        d["subject"], d["session"] = f.parts[-4], f.parts[-3]
        rows.append(d)
    return pd.concat(rows, ignore_index=True)

el = stack("sub-*/ses-*/ieeg/*_electrodes.tsv")
el["AnatomicalLocation"] = el["AnatomicalLocation"].astype(str)
el["gyrus"] = el.AnatomicalLocation.str.split(",").str[0].str.strip()
groups = {"hippocampus": ["Hipp"], "amygdala": ["Amyg"], "parahippocampal": ["PhG"],
          "lateral_temporal": ["STG", "MTG", "ITG", "pSTS"],
          "unlabeled": ["no_label_found", "nan"]}
g2r = {g: r for r, gs in groups.items() for g in gs}
el["region"] = el.gyrus.map(g2r).fillna("other_labeled")
el.to_csv("/kaggle/working/meta/electrodes_all.csv", index=False)

ev = stack("sub-*/ses-*/ieeg/*_events.tsv")
ev["clean"] = ((ev.SetSize > 0) & (ev.Artifact == 0)).astype(int)
ev.to_csv("/kaggle/working/meta/trials_all.csv", index=False)
print("Clean trials:", int(ev.clean.sum()), "of", len(ev))

Clean trials: 3185 of 3353
